# Evaluation Metrics — Spectral Composition (Issues #49 + #50)

This notebook presents every evaluation deliverable produced by
`scripts/run_evaluation.py`, extended per
[issue #50](https://github.com/tuned-org-uk/latent-sound-diffusion/issues/50)
Phase 1:

- **Phase 1** (Tables 1, 2, 6): reconstruction L1 (train/val/test) +
  $\lambda^{ED}$ ablation + per-band spectral energy retention
- **Phase 2** (Table 3): FAD-proxy + CLAP-proxy (graph vs baseline)
- **Phase 3a** (Table 4): dehydration compression ratio vs corpus size $N$
- **Phase 3b** (Table 5): rehydration coherence (MIDI pitch vs spectral
  centroid/rolloff, Pearson $r$)
- **Phase 3c** (CSV + figure): TRUE recursive variant drift — outputs fed
  back through `condition_on_audio` → `synthesize_midi` for R rounds
- **Sweeps** (issue #50): chart rank $q$, latent NOISE_INJECT
  (diversity-vs-fidelity), heat-death $\varepsilon$ (steps-vs-quality)

**Calibration.** The paper introduces a novel workflow — dehydration →
diffusion → rehydration → recursive variation — at preliminary scale
(256-sample NSynth subset, 20 epochs, real frozen EnCodec). These tables
are hypothesis-generating evidence for the paradigm, **not** benchmark
claims: numbers are reported verbatim, the matched baseline contextualises
(and currently outperforms) the graph decoder at this scale, and the
workflow-affordance metrics (compression, coherence, variant drift,
diversity-vs-fidelity) stand alongside fidelity metrics.

By default this notebook **loads the pre-computed results** produced by
the faithful full-training pipeline `scripts/run_evaluation.py`
(256-sample subset, 20 epochs, real frozen EnCodec, MPS with grad
clipping; the graph decoder's per-time-step graph filter makes training
device-stable — see issue #51). Set `TRAIN_FROM_SCRATCH = True` below to
re-run the full pipeline in-place.

> **FAD/CLAP methodology**: `fadtk` and `laion-clap` were removed due to
> dependency conflicts (see README "Open issues" and
> `docs/postmortem-c-spec-regression.md`). Phase 2/3c use dependency-free
> proxies on the same frozen EnCodec feature space the model already
> uses. The methodology is recorded in each CSV (`*_method` columns). See
> `src/ald_sc/eval.py` for the implementations.

## Configuration

In [ ]:
from pathlib import Path
import csv

# Set True to re-run the full training pipeline in-notebook (~20 min CPU).
# False (default) loads the pre-computed results from results/*.csv.
TRAIN_FROM_SCRATCH = False

RESULTS_DIR = Path.cwd().parent / 'results'
DATA_DIR = Path.cwd().parent / 'data'

SEED = 3407
AUDIO_LENGTH = 96000
SAMPLE_RATE = 24000
TEXT_PROMPT = 'warm electronic bass synth'

print(f'Results dir: {RESULTS_DIR}')
print(f'Train from scratch: {TRAIN_FROM_SCRATCH}')

## Helpers

In [ ]:
def show(rows):
    'Lightweight table display (no pandas dependency).'
    if not rows:
        print('(empty)')
        return
    keys = list(rows[0].keys())
    widths = [max(len(str(k)), *(len(str(r.get(k, ''))) for r in rows)) for k in keys]
    hdr = ' | '.join(str(k).ljust(w) for k, w in zip(keys, widths))
    sep = '-+-'.join('-' * w for w in widths)
    print(hdr); print(sep)
    for r in rows:
        print(' | '.join(str(r.get(k, '')).ljust(w) for k, w in zip(keys, widths)))

def load_csv(name):
    path = RESULTS_DIR / name
    if not path.exists():
        print(f'  WARNING: {path} not found (run scripts/run_evaluation.py first)')
        return []
    with open(path) as f:
        return list(csv.DictReader(f))

print('Helpers ready.')

## Phase 1: Reconstruction L1 (Table 1) + $\lambda^{ED}$ Ablation (Table 2)

Controlled comparison: graph decoder (`WaveReconstructionBlock` + $U_q$ +
$\lambda^{ED}$) vs matched-capacity baseline decoder (plain `ResBlock1d`,
no graph structure), both trained on the 256-sample NSynth subset with the
real frozen EnCodec encoder for 20 epochs.

In [ ]:
table1 = load_csv('table1_reconstruction.csv')
print('=== Table 1: Reconstruction L1 (train/val/test) ===')
show(table1)

In [ ]:
table2 = load_csv('table2_ablation.csv')
print('=== Table 2: lambda_ED Ablation (c_spec on vs off) ===')
show(table2)

## Phase 1 (extension): Per-band Spectral Energy Retention (Table 6)

For each spectral mode $k$ of the prior chart: the normalised band
energy of the original audio's EnCodec features vs the reconstruction's
(re-encoded), per split and decoder. ``retention`` = $e^{recon}_k /
e^{orig}_k$ (1.0 = perfect); ``cosine`` = mean cosine similarity between
band-energy vectors (split-level summary). Does the decoder preserve the
chart's spectral energy allocation?

In [ ]:
table6 = load_csv('table6_band_retention.csv')
print('=== Table 6: Per-band spectral energy retention ===')
show(table6[:16])
if len(table6) > 16:
    print(f'  ... ({len(table6) - 16} more rows)')

## Phase 2: FAD-proxy + CLAP-proxy (Table 3)

Generates 16 latents via DDIM, decodes with each decoder, and compares the
resulting audio feature distribution to the held-out test-set EnCodec
features. **FAD-proxy** = Frechet distance over frozen EnCodec pooled
features (lower = generated distribution closer to reference). **CLAP-proxy**
= cosine similarity between the generated bank's mean EnCodec feature and a
deterministic hashing embedding of the text prompt.

In [ ]:
table3 = load_csv('table3_fad_clap.csv')
print('=== Table 3: FAD-proxy + CLAP-proxy (graph vs baseline) ===')
show(table3)

## Phase 3a: Dehydration Compression Ratio (Table 4)

Bits required to represent the raw audio library vs the dehydrated
ArrowSpace prior $(L_F, U_q, \lambda^{ED})$ (stored once) plus per-clip
EnCodec code-rate storage. The prior amortises, so the ratio grows with $N$
and asymptotes to the per-clip code-rate ratio.

In [ ]:
table4 = load_csv('table4_compression.csv')
print('=== Table 4: Compression ratio vs corpus size N ===')
show(table4)

## Phase 3b: Rehydration Coherence (Table 5)

Render an ascending MIDI scale via `synthesize_midi` (Mode C), compute the
spectral-centroid contour of the render and the MIDI pitch contour sampled
at the same frames, and report the Pearson correlation over active-note
frames. A higher $r$ means the rehydrated audio's spectral centroid tracks
the scored pitch contour.

In [ ]:
table5 = load_csv('table5_coherence.csv')
print('=== Table 5: Rehydration coherence (MIDI pitch vs spectral centroid) ===')
show(table5)

## Phase 3c: Recursive Variant Drift (CSV + figure)

**True recursion** (issue #50): round 0 renders a MIDI score from a
freshly generated bank; round $r$ conditions on the previous round's
render (`condition_on_audio`, Mode B), rebuilds the bank, and re-renders
the same score (`synthesize_midi`, Mode C). Reports per round: CLAP-proxy
distance to the round-0 render (cumulative novelty) and centroid/rolloff
drift (spectral character evolution). This replaces the earlier
MIDI-rotation approximation with faithful recursive feeding.

In [ ]:
rec_rows = load_csv('recursive_variants.csv')
print('=== Recursive variant drift (true R-round recursion) ===')
show(rec_rows)

In [ ]:
import matplotlib.pyplot as plt

fig_path = RESULTS_DIR / 'fig_variant_diversity.png'
if fig_path.exists():
    from IPython.display import Image, display
    print('fig_variant_diversity.png:')
    display(Image(filename=str(fig_path)))
else:
    print(f'  {fig_path} not found (run scripts/run_evaluation.py first)')

if rec_rows:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    xs = [int(r['round']) for r in rec_rows]
    dists = [float(r['clap_distance_to_round0']) for r in rec_rows]
    cents = [float(r['centroid_mean_hz']) for r in rec_rows]
    rolls = [float(r['rolloff_mean_hz']) for r in rec_rows]
    ax1.plot(xs, dists, 'o-', color='tab:blue')
    ax1.set_xlabel('Recursion round R')
    ax1.set_ylabel('CLAP-proxy distance to round 0')
    ax1.set_title('Cumulative novelty drift')
    ax1.grid(True, alpha=0.3)
    ax2.plot(xs, cents, 'o-', label='centroid (Hz)', color='tab:orange')
    ax2.plot(xs, rolls, 's-', label='rolloff (Hz)', color='tab:green')
    ax2.set_xlabel('Recursion round R')
    ax2.set_title('Spectral drift over rounds')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
else:
    print('No recursive_variants.csv data to plot.')

## Issue #50 sweeps: $q$, NOISE_INJECT, heat-death $\varepsilon$

- **$q$ sweep** (`sweep_q.csv`): chart rank $q \in \{4, 8, 16, 32\}$ —
  prior rebuilt and graph decoder retrained per $q$; test L1 + band-
  retention cosine. How many spectral modes does the chart need at this
  scale?
- **NOISE_INJECT sweep** (`sweep_noise.csv`): latent noise $\in
  \{0.0, 0.1, 0.25, 0.5\}$ — the workflow's diversity-vs-fidelity
  dial: test L1 (fidelity) vs variant distance (novelty affordance).
- **$\varepsilon$ sweep** (`sweep_eps.csv`): heat-death threshold $\in
  \{10^{-2}, 10^{-3}, 10^{-4}\}$ (sampling only) — intrinsic stopping
  trades steps for quality: mean DDIM steps used vs FAD-proxy.

In [ ]:
sweep_q = load_csv('sweep_q.csv')
print('=== Sweep: chart rank q (decoder retrained per q) ===')
show(sweep_q)

In [ ]:
sweep_noise = load_csv('sweep_noise.csv')
print('=== Sweep: NOISE_INJECT (diversity vs fidelity) ===')
show(sweep_noise)

In [ ]:
sweep_eps = load_csv('sweep_eps.csv')
print('=== Sweep: heat-death epsilon (sampling only) ===')
show(sweep_eps)

if sweep_eps:
    fig, ax = plt.subplots(figsize=(6, 4))
    xs = [float(r['eps']) for r in sweep_eps]
    ys = [float(r['mean_steps']) for r in sweep_eps]
    ax.plot(xs, ys, 'o-')
    ax.set_xscale('log')
    ax.set_xlabel('heat-death epsilon')
    ax.set_ylabel('mean DDIM steps used')
    ax.set_title('Intrinsic stopping: epsilon vs steps')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

## ESC-50 cross-corpus run (tagged tables)

The same 256-subset pipeline on ESC-50 environmental audio (~600 MB,
CC BY 4.0) writes `esc50_`-prefixed tables — cross-corpus evidence that
the workflow applies beyond musical instruments, without clobbering the
NSynth tables.

In [ ]:
esc50_files = ['esc50_table1_reconstruction.csv', 'esc50_table3_fad_clap.csv',
              'esc50_table5_coherence.csv', 'esc50_recursive_variants.csv']
for f in esc50_files:
    rows = load_csv(f)
    if rows:
        print(f'=== {f} ===')
        show(rows)
        print()

## Re-run from scratch (optional)

The full training pipeline (prior construction, graph + baseline decoder
training, DiT training, all metrics + sweeps) is in
`scripts/run_evaluation.py`. To reproduce the CSVs from scratch:

```bash
uv run python scripts/run_evaluation.py --device mps \
    --ablation-q 4 8 16 32 --ablation-noise 0.0 0.1 0.25 0.5 \
    --ablation-eps 1e-2 1e-3 1e-4

# ESC-50 tagged run
uv run python scripts/run_evaluation.py --device mps \
    --data-dir data/esc50/ESC-50-master/audio --tag esc50_
```

Set `TRAIN_FROM_SCRATCH = True` in the configuration cell and re-execute to
run the same pipeline in-notebook. The eval engine itself is in
`src/ald_sc/eval.py` (reusable, unit tests in `tests/test_eval.py`).

## Summary

In [ ]:
print('=== Deliverables ===')
for f in ['table1_reconstruction.csv', 'table2_ablation.csv', 'table3_fad_clap.csv',
          'table4_compression.csv', 'table5_coherence.csv', 'table6_band_retention.csv',
          'recursive_variants.csv', 'fig_variant_diversity.png',
          'sweep_q.csv', 'sweep_noise.csv', 'sweep_eps.csv']:
    p = RESULTS_DIR / f
    print(f'  {"OK" if p.exists() and p.stat().st_size > 0 else "MISSING"}: {p.name}')